In [1]:
from brian2 import *
import os, sys
root = os.path.dirname(os.getcwd())  # go up from 'Interactive Models'
models_dir = os.path.join(root, 'Neuron and Synapse Models')
tools_dir = os.path.join(root, 'Tools')
for p in (models_dir, tools_dir):
    if p not in sys.path:
        sys.path.append(p)

from neuronModels import *
from ringAttractorClass import *
from plottingTools import *
from utils import compute_firing_rate
from boundedRingAttractorClass import BoundedRingAttractor


import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, IntSlider, Dropdown, HTML, FloatText, Label, ToggleButton

# Simulation parameters
defaultclock.dt = 0.1*ms

In [2]:
# Code block to allow easy passing of intital values
values_flag = True
if values_flag:
    init_values = {
        'num_neurons': 120,
        'tau_val': 10,
        'sigma_noise_val': 0.1,
        'stimulus_center': 0.75,
        'stimulus_width': 0.2,
        'I0_val': 30,
        'g_cosine_val': 0.071, #0.058 # 0.11150
        'w_inh_val': -0.68, #-0.72, #  -0.86000
        'velocity_input': 2.0,
        'input_duration': 1.0,
        'duration_val': 1.0,
        'velocity_duration_val': 3.0,
        'upper_bound_neuron': 29
    }

In [3]:
# Create sliders for parameters
num_neurons_slider = IntSlider(
    min=50,
    max=200, 
    step=10, 
    value=init_values['num_neurons'] if values_flag else 120, 
    description='Number of Neurons:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

tau_slider = FloatSlider(
    min=1, 
    max=20, 
    step=1, 
    value=init_values['tau_val'] if values_flag else 10, 
    description='Tau (ms):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_noise_slider = FloatSlider(
    min=0.1, 
    max=5, 
    step=0.1, 
    value=init_values['sigma_noise_val'] if values_flag else 1, 
    description='Noise Sigma (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_center_slider = FloatSlider(
    min=0, 
    max=2*pi, 
    step=0.01, 
    value=init_values['stimulus_center'] if values_flag else 0, 
    description='Stimulus Center (rad):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_width_slider = FloatSlider(
    min=0.1, 
    max=2.0, 
    step=0.01, 
    value=init_values['stimulus_width'] if values_flag else 0.5, 
    description='Stimulus Width:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

I0_slider = FloatSlider(
    min=0, 
    max=150, 
    step=5, 
    value=init_values['I0_val'] if values_flag else 30, 
    description='Input Amplitude (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

g_cosine_slider = FloatSlider(
    min=0.005,
    max=1.0,
    step=0.0001,
    value=init_values['g_cosine_val'] if values_flag else 1.0,  # conditional value for cosine gain
    description='Gain Cosine (mV):',
    continuous_update=False,
    style={'description_width': '150px'},
    readout_format='.5f'
)

w_inh_slider = FloatSlider(
    min=-1.0, 
    max=-0.01, 
    step=0.01, 
    value=init_values['w_inh_val'] if values_flag else -0.6, 
    description='Global Inhibition (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
    readout_format='.5f'
)

velocity_input_slider = FloatSlider(
    min=-8.0, 
    max=8.0, 
    step=0.1, 
    value=init_values['velocity_input'] if values_flag else 0.0, 
    description='Velocity Input (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)



duration_box = FloatText(
    value=init_values['duration_val'] if values_flag else 2,
    description='In-Between Duration (s):',
    style={'description_width': '150px'}
)

input_duration_box = FloatText(
    value=init_values['input_duration'] if values_flag else 1,
    description='Input Duration (s):',
    style={'description_width': '150px'}
)

velocity_duration_box = FloatText(
    value=init_values['velocity_duration_val'] if values_flag else 0.5,
    description='Velocity Duration (s):',
    style={'description_width': '150px'}
)

angles_dropdown = Dropdown(
    options=[ 'degrees', 'radians', 'radians (symbolic)'],
    value='degrees',
    description='Angular Representation:',
    style={'description_width': '150px'},
)

autapse_button = ToggleButton(
    value = True,
    description='Autapse',
    tooltip='Allows autapse connections',
    button_style=''
)

global_inh_button = ToggleButton(
    value = True,
    description='Global Inhibitory Neuron',
    tooltip='Adds a global inhibitory neuron',
    button_style='',
    layout = Layout(width='auto', height='auto')
)

upper_bound_neuron_slider = IntSlider(
    min=1,
    max=num_neurons_slider.max,
    step=1,
    value=init_values['upper_bound_neuron'] if values_flag else 20,
    description='Upper Bound Neuron:',
    continuous_update=False,
    style={'description_width': '150px'}
)

# Create dictionary of widgets
widgets = {
    'num_neurons': num_neurons_slider,
    'tau_val': tau_slider,
    'sigma_noise_val': sigma_noise_slider,
    'stimulus_center': stimulus_center_slider,
    'stimulus_width': stimulus_width_slider,
    'I0_val': I0_slider,
    'g_cosine_val': g_cosine_slider,
    'duration_val': duration_box,
    'input_duration_val': input_duration_box,
    'velocity_duration_val': velocity_duration_box,
    'ticks_angles': angles_dropdown,
    'autapse': autapse_button,
    'global_inh': global_inh_button,
    'velocity_input': velocity_input_slider,
    'w_inh_val': w_inh_slider,
    'upper_bound_neuron': upper_bound_neuron_slider
}

In [4]:
import numpy as np
import os

# Updated interactive_simulator to accept velocity_duration_val
def interactive_simulator(num_neurons, tau_val, sigma_noise_val,
                          stimulus_center, stimulus_width, I0_val, 
                          g_cosine_val, velocity_input, w_inh_val,
                          duration_val, input_duration_val, velocity_duration_val,
                          autapse, global_inh, ticks_angles, upper_bound_neuron):
    
    glob_inh_flag = global_inh   # use value from widget
    
    # Clear any previous figures
    plt.close('all')
    
    # Convert slider values to Brian units
    tau = tau_val * ms
    sigma_noise = sigma_noise_val * mV
    V_rest = -70 * mV
    I0 = I0_val * mV
    sim_duration = duration_val*second
    g_cosine = g_cosine_val * mV
    w_inh_v = w_inh_val * mV
    velocity_duration = velocity_duration_val  # Use the parameter from the widget
    
    # Define neuron positions
    positions = linspace(0, 2*pi, num_neurons, endpoint=False)
    
    # Calculate external input
    d = np.angle(np.exp(1j * (positions - stimulus_center)))
    I0_CONST=40*mV
    I_ext_array = I0 * np.exp(-(d**2) / (2 * stimulus_width**2))+I0_CONST
    
    # Set up neuron model
    neuron_eq = Equations(LIF_xi_vel_eq, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise)
    
    # Set up ring attractor
    Vth = -48 * mV
    V_reset = -80 * mV
    refractory_period = 5 * ms
    
    joint_limit=upper_bound_neuron/num_neurons*2*pi

    
    # Create the ring attractor network
    ringAttractor = BoundedRingAttractor(neuron_eq, 
                         num_neurons, 
                         Vth, V_reset, refractory_period,
                         autapse=autapse, limit_neuron=upper_bound_neuron,
                         glob_inh=glob_inh_flag, w_inh=w_inh_v,
                         g_cosine=g_cosine)

    # Set external input
    ringAttractor.ring_pool.I_ext = I_ext_array
    ringAttractor.ring_pool.I_vel = 0.0*volt
    
    t_thresh=ringAttractor.t_thresh 
    theta_star=ringAttractor.theta_star
    
        
    # Setup monitors
    spikemon = SpikeMonitor(ringAttractor.ring_pool)
    statemon = StateMonitor(ringAttractor.ring_pool, 'V', record=True)
    inputmon = StateMonitor(ringAttractor.ring_pool, 'I_ext', record=True)
    
    #+---------------------------------------------------------------------------+
    #|                             Network Operations                            |
    #+---------------------------------------------------------------------------+
    # Clipping - Reverse Potential Behaviour: Define a network operation to enforce the lower bound on the membrane potential
    @network_operation(dt=defaultclock.dt)
    def enforce_lower_bound():
        # Using the built-in clip function (from numpy)
        ringAttractor.ring_pool.V[:] = clip(ringAttractor.ring_pool.V[:], V_reset, inf*volt)
    
    # Set of Brian objects to be added to the network
    localObjects = [enforce_lower_bound,
                    spikemon, statemon, inputmon]
    
    if glob_inh_flag:
        statemon_inh = StateMonitor(ringAttractor.glob_inh_neuron, 'V', record=True)
        spikemon_inh = SpikeMonitor(ringAttractor.glob_inh_neuron)
        localObjects.extend([statemon_inh, spikemon_inh])
        
    
    net = Network(ringAttractor.BrianObjects+localObjects)
    
    input_on = input_duration_val * second
    input_off = sim_duration
    velocity_on = velocity_duration * second
    end_duration = sim_duration
    
    total_duration = input_on + input_off + velocity_on + end_duration

    # Run simulation
    net.run(input_on)
    
    # Turn off input for the second half
    ringAttractor.ring_pool.I_ext = I0_CONST
    net.run(input_off)
    
    # Turn on velocity input
    ringAttractor.ring_synapses_asym.vel_in = velocity_input
      
    net.run(velocity_on)
    
    # Turn off velocity input
    ringAttractor.ring_synapses_asym.vel_in = 0.0
    
    # Turn off velocity input and run for the rest of the duration
    net.run(end_duration)
    
    #+---------------------------------------------------------------------------+
    #|                           Plotting the Results                            |
    #+---------------------------------------------------------------------------+
    # Use a 3x2 grid so the time-resolved PVA has its own row slot.
    fig = plt.figure(figsize=(14, 14))
    
    # 1. Input Current Plot (row1 col1)
    ax1 = fig.add_subplot(3, 2, 1)
    ax1.plot(positions/(2*pi), I_ext_array/mV)
    ax1.set_title('Input Current')
    ax1.set_xlabel('Position (fraction of 2π)')
    ax1.set_ylabel('Current (mV)')
    
    # 2. Raster Plot (row1 col2)
    ax2 = fig.add_subplot(3, 2, 2)
    ax2.axvspan((input_on+input_off)/second, (input_on+velocity_on+input_off)/second, color='green', alpha=0.2, label='Velocity Input ON')

    raster_plot(spikemon, ax=ax2, stim_periods=(0*second, input_on),
                stim_display_method='highlight', duration=total_duration, num_neurons=num_neurons, y_axisFull=True)
    ax2.axhline(y=ringAttractor.limit_neuron, color='blue', linestyle='--', linewidth=2, label='upper bound neuron')
    ax2.axhline(y=ringAttractor.neuron_at_theta_star, color='red', linestyle='--', linewidth=2, label='theta_star')
    ax2.axhline(y=0, color='green', linestyle='--', linewidth=2, label='lower bound neuron')
    def angle_to_neuron(angle):
        return int((angle % (2 * np.pi)) / (2 * np.pi) * num_neurons)
    if velocity_input<0:
        ax2.axhspan(angle_to_neuron(joint_limit - t_thresh), angle_to_neuron(joint_limit + t_thresh), color='blue', alpha=0.2)
        ax2.axhspan(angle_to_neuron(theta_star - t_thresh),angle_to_neuron( theta_star + t_thresh), color='red', alpha=0.2)
    elif velocity_input>0:
        ax2.axhspan(0, angle_to_neuron(0 + t_thresh), color='green', alpha=0.2)
        ax2.axhspan(angle_to_neuron(2*pi-t_thresh), num_neurons, color='green', alpha=0.2)
        ax2.axhspan(angle_to_neuron(theta_star - t_thresh), angle_to_neuron(theta_star + t_thresh), color='red', alpha=0.2)
    ax2.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), fontsize=8)
    
    # 3. Firing Rate Profile Plot (row2 col1)
    ax3 = fig.add_subplot(3, 2, 3)
    start_mask=input_on+input_off+velocity_on+input_off/2
    mask=spikemon.t>start_mask
    reduced_duration=total_duration-start_mask
    class ReducedMonitor:
        def __init__(self, t=None, i=None):
            self.t = t
            self.i = i
    reduced_monitor = ReducedMonitor(t=spikemon.t[mask], i=spikemon.i[mask])
    firing_rate, _ = firing_rate_profile(reduced_monitor, positions/(2*pi), end_duration, ax=ax3)
    
    # 4. Polar Plot of PVA (row2 col2)
    ax4 = fig.add_subplot(3, 2, 4, projection='polar')
    polar_plot_PVA(firing_rate, positions, scale=1.2, ax=ax4)
    pva_angle, pva_magnitude = calculate_PVA(firing_rate, positions)
    joint_limit = upper_bound_neuron / num_neurons * 2 * pi
    ax4.set_rlim(0, second*1/refractory_period)
    ax4.plot([0, 0], [0, ax4.get_rmax()], color='green', linestyle='--', linewidth=2, label='Theta=0')
    ax4.plot([joint_limit, joint_limit], [0, ax4.get_rmax()], color='blue', linestyle='--', linewidth=2, label=f'Theta=joint_limit= {joint_limit*180/pi:.2f}')
    ax4.plot([ringAttractor.theta_star,ringAttractor.theta_star], [0, ax4.get_rmax()], color='red', linestyle='--', linewidth=2, label=f'Theta=theta_star= {ringAttractor.theta_star*180/pi:.2f}')
    ax4.plot([pva_angle,pva_angle], [0,pva_magnitude], 'cyan', label=f'PVA: ({pva_angle*180/pi:.2f}, {pva_magnitude:.2f})')
    ax4.legend(loc='upper right', bbox_to_anchor=(1.15, 1.1), fontsize=8)
    
    # 5. Time-Resolved PVA Plot (row3 spans both columns)
    ax5 = fig.add_subplot(3, 1, 3)  # full-width axis on third row
    _ , _ = time_resolved_PVA(spikemon, positions, total_duration, num_neurons, window_size=100*ms,
                              ax=ax5, color_windows=False, stim_periods=(0*second, input_on))
    ax5.set_title('Time-Resolved PVA')
    # Overlay boundary/threshold annotations (converted to angles)
    angle_per_index = 2 * pi / ringAttractor.N
    limit_angle = ringAttractor.limit_neuron * angle_per_index
    theta_star_angle = ringAttractor.theta_star  # already an angle
    lower_angle = 0.0

    # Horizontal reference lines
    ax5.axhline(limit_angle, color='blue', linestyle='--', linewidth=2, label='upper bound neuron (angle)')
    ax5.axhline(theta_star_angle, color='red', linestyle='--', linewidth=2, label='theta_star (angle)')
    ax5.axhline(lower_angle, color='green', linestyle='--', linewidth=2, label='lower bound neuron (angle)')

    # Highlight angular threshold regions
    if velocity_input < 0:
        # Joint limit band
        jl_low = max(joint_limit - t_thresh, 0)
        jl_high = min(joint_limit + t_thresh, 2 * pi)
        ax5.axhspan(jl_low, jl_high, color='blue', alpha=0.15)
        # Theta star band
        ts_low = max(theta_star - t_thresh, 0)
        ts_high = min(theta_star + t_thresh, 2 * pi)
        ax5.axhspan(ts_low, ts_high, color='red', alpha=0.15)
    elif velocity_input > 0:
        # Wrap-around near 0
        ax5.axhspan(0, min(t_thresh, 2 * pi), color='green', alpha=0.15)
        # Wrap-around near 2π
        ax5.axhspan(max(2 * pi - t_thresh, 0), 2 * pi, color='green', alpha=0.15)
        # Theta star band
        ts_low = max(theta_star - t_thresh, 0)
        ts_high = min(theta_star + t_thresh, 2 * pi)
        ax5.axhspan(ts_low, ts_high, color='red', alpha=0.15)

    ax5.set_ylabel('Angle (rad)')
    # Ensure full angular range visible
    ax5.set_ylim(0, 2 * pi)

    # Consolidate legend (avoid duplicates)
    handles, labels = ax5.get_legend_handles_labels()
    seen = {}
    uniq_h, uniq_l = [], []
    for h, l in zip(handles, labels):
        if l not in seen:
            seen[l] = True
            uniq_h.append(h)
            uniq_l.append(l)
    ax5.legend(uniq_h, uniq_l, loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()
        
    # Visualize asymmetric connectivity for theta_pre mid-point (optional diagnostics)
    theta_pre = joint_limit/2
    joint_limit_val = joint_limit  # already in radians
    # (Further diagnostic plotting removed for layout clarity)
    
    
    
    

In [5]:
# # Define common style and layout settings for sliders
# common_style = {'description_width': '150px'}
# common_layout = Layout(width='300px', margin='10px auto')

# Create the parameter boxes with descriptive titles
from ipywidgets import Button, Output  # add missing widgets

box_neurons = VBox([
    HTML(value="<b>Neurons Parameters:</b>"),
    num_neurons_slider,
    upper_bound_neuron_slider,
    tau_slider,
    sigma_noise_slider
], layout=Layout(width='25%', align_items='center'))

box_input = VBox([
    HTML(value="<b>Input Parameters:</b>"),
    stimulus_center_slider,
    stimulus_width_slider,
    I0_slider,
    velocity_input_slider
], layout=Layout(width='25%', align_items='center'))

checkbox_subBox = HBox([
    autapse_button,
    global_inh_button
], layout=Layout(align_items='center'))

box_connectivity = VBox([
    HTML(value="<b>Connectivity Profile Parameters:</b>"),
    checkbox_subBox,
], layout=Layout(width='25%', align_items='center'))

# Rework connectivity box updater to always reflect current global_inh and show relevant sliders
# (Currently only g_cosine and w_inh are relevant; logic placeholder if more profiles reintroduced.)
def update_connectivity_box(*args):
    children = [HTML(value="<b>Connectivity Profile Parameters:</b>"), checkbox_subBox]
    # Always show cosine gain slider for now
    children.append(g_cosine_slider)
    if global_inh_button.value:
        children.append(w_inh_slider)
    box_connectivity.children = children

# Observe changes that affect connectivity panel
global_inh_button.observe(update_connectivity_box, names='value')
# Initial population
update_connectivity_box()

box_simulation = VBox([
    HTML(value="<b>Simulation Parameters:</b>"),
    input_duration_box,
    velocity_duration_box,
    duration_box,
    angles_dropdown
], layout=Layout(width='25%', align_items='center'))

controls = HBox(
    [box_neurons, box_input, box_connectivity, box_simulation],
    layout=Layout(justify_content='center', margin='20px')
)

# Manual execution button + output area
run_button = Button(
    description='Run Simulation',
    button_style='success',  # green style (builtin)
    icon='play',
    layout=Layout(width='50%', height='55px')
)
run_button.style.font_weight = 'bold'
run_button.style.font_size = '20px'

output_area = Output(layout=Layout(border='1px solid #ccc', padding='10px'))

# Keep upper bound neuron slider in range when number of neurons changes
def on_num_neurons_change(change):
    new_n = change['new']
    # Ensure upper bound slider respects new max - 1 (cannot point to non-existent index)
    upper_bound_neuron_slider.max = new_n - 1
    if upper_bound_neuron_slider.value >= new_n:
        upper_bound_neuron_slider.value = new_n - 1

num_neurons_slider.observe(on_num_neurons_change, names='value')

# Click handler to execute simulation ONLY when button is pressed
def on_run_clicked(b):
    run_button.disabled = True
    try:
        with output_area:
            output_area.clear_output(wait=True)
            print("Running simulation with current parameters...")
            interactive_simulator(
                num_neurons_slider.value,
                tau_slider.value,
                sigma_noise_slider.value,
                stimulus_center_slider.value,
                stimulus_width_slider.value,
                I0_slider.value,
                g_cosine_slider.value,
                velocity_input_slider.value,
                w_inh_slider.value,
                duration_box.value,
                input_duration_box.value,
                velocity_duration_box.value,
                autapse_button.value,
                global_inh_button.value,
                angles_dropdown.value,
                upper_bound_neuron_slider.value
            )
            print("Simulation complete.")
    finally:
        run_button.disabled = False

run_button.on_click(on_run_clicked)

# Assemble dashboard (button on top)
dashboard = VBox([
    HBox([run_button], layout=Layout(justify_content='center')),
    controls,
    output_area
], layout=Layout(align_items='stretch', justify_content='flex-start'))

display(dashboard)